# Basins attributes

In [ ]:
import pandas as pd
import numpy as np
import os

import xarray as xr
import geopandas as gpd
import rioxarray as rio
import xrspatial as xrs
from exactextract import exact_extract

from andeangc.config import find_repo_root, get_dir, get, get_version_dir

inputs  = get('inputs')
outputs = get('outputs')
wd = find_repo_root()
os.chdir(wd)

path_final = get_version_dir('final')

# local modules
from andeangc import polygon_extract

## AndeanGC data

In [ ]:
AndeanGC_metadata = pd.read_csv(path_final / 'AndeanGC_metadata.csv')
AndeanGC_shape    = gpd.read_file(path_final / 'AndeanGC_shape.gpkg')

## Get features

In [ ]:
# basic attributes
AndeanGC_shape_projected = AndeanGC_shape.to_crs(get('epsg_utm'))  # UTM zone for Andes
AndeanGC_shape["basin_cenlat"] = AndeanGC_shape_projected.centroid.to_crs(4326).y
AndeanGC_shape["basin_cenlon"] = AndeanGC_shape_projected.centroid.to_crs(4326).x
AndeanGC_shape["total_perim"] = AndeanGC_shape_projected.geometry.length / 1e3  # from m to km
AndeanGC_shape["gc_index"] = AndeanGC_shape.total_perim/(2*(np.sqrt(np.pi*AndeanGC_shape.basin_area)))

In [ ]:
# topographic attributes (based on FABDEM)
dem = xr.open_dataset(get_dir('dem') / inputs['dem_attributes']).band_data.sel(band=1, drop = True)
dem = dem.sel(x=slice(AndeanGC_shape.total_bounds[0], AndeanGC_shape.total_bounds[2]), 
              y=slice(AndeanGC_shape.total_bounds[3], AndeanGC_shape.total_bounds[1]))
dem  = dem.fillna(0)  # NAs to sea level (= 0)

slope  = xrs.slope(dem.rio.reproject(f"EPSG:{get('epsg_utm')}"))
slope  = slope.rio.reproject(f"EPSG:{get('epsg_wgs84')}")
aspect = xrs.aspect(dem)

AndeanGC_shape = polygon_extract.extract_attributes(dem, AndeanGC_shape, "elev_mean",    fun = "mean")
AndeanGC_shape = polygon_extract.extract_attributes(dem, AndeanGC_shape, "elev_median",  fun = "median")
AndeanGC_shape = polygon_extract.extract_attributes(dem, AndeanGC_shape, "elev_std",     fun = "stdev")
AndeanGC_shape = polygon_extract.extract_attributes(slope, AndeanGC_shape, "slope_mean",  fun = "mean")
AndeanGC_shape = polygon_extract.extract_attributes(aspect, AndeanGC_shape, "aspect_mean", fun = "mean")

In [ ]:
# climate attributes (ERA5)
ref_period = get('period_ref_climate')

# Load all climate data into a single stack
climate_stack = xr.open_mfdataset(str(get_dir('era5') / f"*_ERA5_{get('period_climate')[0]}_{get('period_climate')[1]}.nc"))
climate_stack = climate_stack.sel(time=slice(ref_period[0], ref_period[1]))
climate_stack['tas'] = (climate_stack.tasmax + climate_stack['tasmin']) / 2

# prcp_mean
stack = climate_stack.prcp.resample(time='YS').sum(dim='time').mean(dim='time')
AndeanGC_shape = polygon_extract.extract_attributes(stack, AndeanGC_shape, "prcp_mean_ERA5",  fun = "mean")

# prcp_pci
stack = climate_stack.prcp.resample(time='MS').sum(dim='time').groupby('time.month').mean(dim='time')
stack = (stack ** 2).sum(dim='month') * 100 / (stack.sum(dim='month') ** 2)
AndeanGC_shape = polygon_extract.extract_attributes(stack, AndeanGC_shape, "prcp_pci_ERA5", fun = "mean")

# ev_mean and aridity
stack = climate_stack.ep.resample(time='YS').sum(dim='time').mean(dim='time')
AndeanGC_shape = polygon_extract.extract_attributes(stack, AndeanGC_shape, "ev_mean_ERA5", fun = "mean")
AndeanGC_shape['aridity_ERA5'] = AndeanGC_shape['ev_mean_ERA5'] / AndeanGC_shape['prcp_mean_ERA5']

# solid prcp (< 0◦C)
stack = xr.where(climate_stack.tas > 0, 0, climate_stack.prcp)
stack = stack.resample(time='YS').sum(dim='time').mean(dim='time')
AndeanGC_shape = polygon_extract.extract_attributes(stack, AndeanGC_shape, "solid_prcp_ERA5", fun = "mean")
AndeanGC_shape['frac_snow_ERA5'] = AndeanGC_shape['solid_prcp_ERA5'] / AndeanGC_shape['prcp_mean_ERA5']

# prcp freq
stack = (climate_stack.prcp < 1).sum(dim='time')
stack = climate_stack.prcp.sizes['time'] / stack
AndeanGC_shape = polygon_extract.extract_attributes(stack, AndeanGC_shape, "low_prec_freq_ERA5", fun = "mean")

stack = (climate_stack.prcp > climate_stack.prcp.mean(dim='time') * 5).sum(dim='time')
stack = climate_stack.prcp.sizes['time'] / stack
AndeanGC_shape = polygon_extract.extract_attributes(stack, AndeanGC_shape, "high_prec_freq_ERA5", fun = "mean")

In [ ]:
# glacier attributes

# dhdt in m/yr
dhdt = rio.open_rasterio(get_dir('glacier_dhdt') / outputs['glacier_dhdt']).sel(band=1, drop = True) # in m/yr
dhdt = dhdt.sel(x=slice(AndeanGC_shape.total_bounds[0], AndeanGC_shape.total_bounds[2]), 
                y=slice(AndeanGC_shape.total_bounds[3], AndeanGC_shape.total_bounds[1]))
AndeanGC_shape = polygon_extract.extract_attributes(dhdt, AndeanGC_shape, "glacier_dhdt", fun = "mean")

# vol in km3 (M22)
vol = rio.open_rasterio(get_dir('glacier_thickness') / outputs['glacier_volume_millan']).sel(band=1, drop = True) # in km3
vol = vol.sel(x=slice(AndeanGC_shape.total_bounds[0], AndeanGC_shape.total_bounds[2]), 
               y=slice(AndeanGC_shape.total_bounds[3], AndeanGC_shape.total_bounds[1]))
AndeanGC_shape = polygon_extract.extract_attributes(vol, AndeanGC_shape, "glacier_volume_M22", fun = "sum")

# volume in km3 (F19)
vol = rio.open_rasterio(get_dir('glacier_thickness') / outputs['glacier_volume_farinotti']).sel(band=1, drop = True) # in km3
vol = vol.sel(x=slice(AndeanGC_shape.total_bounds[0], AndeanGC_shape.total_bounds[2]), 
               y=slice(AndeanGC_shape.total_bounds[3], AndeanGC_shape.total_bounds[1]))
AndeanGC_shape = polygon_extract.extract_attributes(vol, AndeanGC_shape, "glacier_volume_F19", fun = "sum")

In [ ]:
# land cover features
land_cover_data = rio.open_rasterio(get_dir('land_cover') / inputs['land_cover'])
land_cover_data = land_cover_data.sel(band=1, drop = True)
land_cover_data = land_cover_data.sel(x=slice(AndeanGC_shape.total_bounds[0], AndeanGC_shape.total_bounds[2]), 
                                      y=slice(AndeanGC_shape.total_bounds[3], AndeanGC_shape.total_bounds[1]))

# collapse the detailed forest classes onto the single forest code
lc_classes = get('land_cover_classes')
lc_forest  = get('land_cover_forest_range')
land_cover_data = land_cover_data.where(
    ~((land_cover_data >= lc_forest[0]) & (land_cover_data <= lc_forest[1])), lc_classes['forest_cover'])
land_cover_data = exact_extract(land_cover_data, AndeanGC_shape, ["unique", "frac"], progress=False, 
                                output = "pandas", include_cols=["gauge_id"])

land_cover_data = land_cover_data.apply(lambda row: pd.Series(dict(zip(row['unique'], row['frac']))), axis=1)
land_cover_data = land_cover_data.set_index(AndeanGC_shape.gauge_id).fillna(0) * 100

AndeanGC_shape = AndeanGC_shape.set_index("gauge_id")
for column, code in lc_classes.items():
    AndeanGC_shape[column] = land_cover_data[code]

In [ ]:
# dams 
dams_data = gpd.read_file(get_dir('dams') / inputs['dams']) 
dams_data = dams_data[dams_data.Capac > get('dam_capacity_threshold')] # only dams with capacity larger than 100 hm3

dams_count = gpd.sjoin(dams_data, AndeanGC_shape.reset_index(), how='inner', predicate='within')
dams_count = dams_count.groupby('gauge_id').size()
AndeanGC_shape = AndeanGC_shape.join(dams_count.rename("dams_count"))
AndeanGC_shape["dams_count"] = AndeanGC_shape["dams_count"].fillna(0)

## Save (and overwrite) data


In [ ]:
# Drop geometry and select only new columns not in AndeanGC_metadata
AndeanGC_shape_subset = AndeanGC_shape.drop(columns='geometry')
AndeanGC_shape_subset = AndeanGC_shape_subset.loc[:, ~AndeanGC_shape_subset.columns.isin(AndeanGC_metadata.columns)]
AndeanGC_metadata = AndeanGC_metadata.join(AndeanGC_shape_subset, on='gauge_id', how='left')

# Save outputs
AndeanGC_shape.to_file(path_final / 'AndeanGC_shape.gpkg')
AndeanGC_metadata.round(4).to_csv(path_final / 'AndeanGC_metadata.csv', index=False)

In [ ]:
# TODO leaf area index (LAI) -> this is missing
#lai_data = rxr.open_rasterio(path_data_raw + "GIS/LAI_MOD15A2H_climatology.nc")
#lai_data = lai_data.mean(dim="time", skipna=True)
#lai_data = exact_extract(lai_data, basin_shp, "mean", progress=False)
#basin_shp["lai_max"] = lai_data.max(axis=1, skipna=True)
#basin_shp["lai_diff"] = lai_data.max(axis=1, skipna=True) - lai_data.min(axis=1, skipna=True)